# Disease → Chemical Association Model (restructured)

Rebuild of the original 62-cell notebook. Key changes vs. the old version:

1. **Direction reframed: disease → chemical.** Given a disease, the model ranks
   every chemical by how likely it is to be associated. (Old version predicted a
   single pair label and was hard to use for ranking.)
2. **Real chemical features.** Each chemical is described by PubChem molecular
   descriptors (MW, XLogP, TPSA, H-bond donors/acceptors, rotatable bonds, heavy
   atoms, complexity) instead of just its one-hot identity. This is the fix for
   the low accuracy — the old model could only memorise IDs, and 82% of
   chemicals appear with only one disease, so it had nothing to generalise from.
3. **Honest evaluation by *grouped* split on chemical.** Test chemicals are never
   seen in training, so the score reflects real generalisation to new chemicals.
4. **Ranking metrics** (precision@k / recall@k per disease) that match how the
   website will actually use the model.
5. **Artifacts exported** for the FastAPI backend.

Run top-to-bottom in Google Colab.


In [ ]:
# 1. Install + imports
!pip install -q xgboost scikit-learn pandas numpy requests

import os, time, json, requests
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
# 2. Load data
from google.colab import drive
drive.mount('/content/drive')

CSV = '/content/drive/MyDrive/AI Projects/Respiratory Diseases - Sheet1.csv'  # <-- adjust path
df = pd.read_csv(CSV)
df.columns = [c.strip().lstrip('#').strip() for c in df.columns]   # ChemicalName, ChemicalID, CasRN, DiseaseName, ...
for c in ['DiseaseName','ChemicalName','ChemicalID','CasRN']:
    df[c] = df[c].astype(str).str.strip()
df = df.drop_duplicates(['DiseaseName','ChemicalID']).reset_index(drop=True)
print(df.shape, '| diseases:', df.DiseaseName.nunique(), '| chemicals:', df.ChemicalID.nunique())

In [ ]:
# 3. Resolve each chemical to a PubChem CID (CAS first, then name), with caching.
#    Cached so this slow step runs only once.
CACHE = '/content/drive/MyDrive/AI Projects/chemical_cid_map.csv'
PUG = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'

def _get(url):
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None

def resolve_cid(name, cas):
    # try CAS registry xref, then exact name
    if cas and cas.lower() not in ('nan',''):
        j = _get(f'{PUG}/compound/xref/RegistryID/{requests.utils.quote(cas)}/cids/JSON')
        if j and j.get('IdentifierList',{}).get('CID'):
            return j['IdentifierList']['CID'][0]
    if name and name.lower() not in ('nan',''):
        j = _get(f'{PUG}/compound/name/{requests.utils.quote(name)}/cids/JSON')
        if j and j.get('IdentifierList',{}).get('CID'):
            return j['IdentifierList']['CID'][0]
    return None

chem = df[['ChemicalID','ChemicalName','CasRN']].drop_duplicates('ChemicalID').reset_index(drop=True)

if os.path.exists(CACHE):
    cid_map = pd.read_csv(CACHE, dtype=str)
else:
    cids = []
    for i, row in chem.iterrows():
        cids.append(resolve_cid(row['ChemicalName'], row['CasRN']))
        if (i+1) % 25 == 0:
            print(f'{i+1}/{len(chem)} resolved'); time.sleep(0.2)
    cid_map = chem.copy(); cid_map['CID'] = cids
    cid_map.to_csv(CACHE, index=False)

cid_map['CID'] = pd.to_numeric(cid_map['CID'], errors='coerce')
print('resolved CIDs:', cid_map['CID'].notna().sum(), '/', len(cid_map))

In [ ]:
# 4. Batch-fetch PubChem descriptors for the resolved CIDs (200 per request)
PROPS = ['MolecularWeight','XLogP','TPSA','HBondDonorCount','HBondAcceptorCount',
         'RotatableBondCount','HeavyAtomCount','Complexity']
FEAT_CACHE = '/content/drive/MyDrive/AI Projects/chemical_features.csv'

if os.path.exists(FEAT_CACHE):
    feats = pd.read_csv(FEAT_CACHE)
else:
    valid = cid_map.dropna(subset=['CID']).copy()
    valid['CID'] = valid['CID'].astype(int)
    rows = []
    ids = valid['CID'].astype(str).tolist()
    for k in range(0, len(ids), 200):
        chunk = ','.join(ids[k:k+200])
        url = f"{PUG}/compound/cid/{chunk}/property/{','.join(PROPS)}/CSV"
        rows.append(pd.read_csv(url)); time.sleep(0.2)
    props_df = pd.concat(rows, ignore_index=True)
    feats = valid.merge(props_df, on='CID', how='left')
    feats.to_csv(FEAT_CACHE, index=False)

print(feats.shape)
feats.head()

In [ ]:
# 5. Build the full (disease x chemical) training grid with real features.
#    Label = 1 if the association exists in the CSV, else 0.
diseases = sorted(df.DiseaseName.unique())
pos = set(zip(df.DiseaseName, df.ChemicalID))

base = feats.set_index('ChemicalID')
grid = []
for d in diseases:
    for cid_chem, frow in base.iterrows():
        grid.append((d, cid_chem))
grid = pd.DataFrame(grid, columns=['DiseaseName','ChemicalID'])
grid['label'] = [int((d,c) in pos) for d,c in zip(grid.DiseaseName, grid.ChemicalID)]
grid = grid.merge(feats, on='ChemicalID', how='left')

# Median-impute missing descriptors (chemicals PubChem couldn't resolve / metals etc.)
for p in PROPS:
    grid[p] = pd.to_numeric(grid[p], errors='coerce')
    grid[p] = grid[p].fillna(grid[p].median())

# Disease one-hot + chemical descriptors = feature matrix
dis_oh = pd.get_dummies(grid['DiseaseName'], prefix='dis')
X = pd.concat([dis_oh, grid[PROPS]], axis=1)
y = grid['label'].values
groups = grid['ChemicalID'].values   # group by chemical for honest split
feature_cols = X.columns.tolist()
print('X:', X.shape, '| positives:', y.sum(), '/', len(y))

In [ ]:
# 6. Honest evaluation: grouped CV so TEST chemicals are unseen in training.
#    Compare against the OLD random-pair split to expose the inflation.
def fit_xgb(Xtr, ytr, Xva, yva):
    spw = (ytr==0).sum() / max((ytr==1).sum(),1)
    m = XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
        min_child_weight=2, objective='binary:logistic',
        eval_metric='aucpr', scale_pos_weight=spw,
        tree_method='hist', random_state=42)
    m.fit(Xtr, ytr, eval_set=[(Xva,yva)], verbose=False)
    return m

# --- grouped (honest) ---
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
aucs, aps = [], []
for tr, te in sgkf.split(X, y, groups):
    m = fit_xgb(X.iloc[tr], y[tr], X.iloc[te], y[te])
    p = m.predict_proba(X.iloc[te])[:,1]
    aucs.append(roc_auc_score(y[te], p)); aps.append(average_precision_score(y[te], p))
print(f'GROUPED-by-chemical (honest):  ROC-AUC {np.mean(aucs):.3f} | PR-AUC {np.mean(aps):.3f}')

# --- random pair split (old, optimistic) ---
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(5, shuffle=True, random_state=42); a2,p2=[],[]
for tr,te in skf.split(X,y):
    m=fit_xgb(X.iloc[tr],y[tr],X.iloc[te],y[te]); pr=m.predict_proba(X.iloc[te])[:,1]
    a2.append(roc_auc_score(y[te],pr)); p2.append(average_precision_score(y[te],pr))
print(f'RANDOM-pair split (optimistic): ROC-AUC {np.mean(a2):.3f} | PR-AUC {np.mean(p2):.3f}')
print('\nThe grouped number is the one to trust/report.')

In [ ]:
# 7. Ranking quality per disease: precision@k / recall@k on held-out chemicals.
#    This matches the website use-case ("top candidate chemicals for disease X").
def ranking_report(ks=(10,20,50)):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    tr, te = next(gss.split(X, y, groups))
    m = fit_xgb(X.iloc[tr], y[tr], X.iloc[te], y[te])
    test = grid.iloc[te].copy(); test['score'] = m.predict_proba(X.iloc[te])[:,1]
    out = []
    for d, g in test.groupby('DiseaseName'):
        g = g.sort_values('score', ascending=False); npos = int(g.label.sum())
        if npos == 0: continue
        row = {'disease': d, 'test_chems': len(g), 'actual_pos': npos}
        for k in ks:
            topk = g.head(k)
            row[f'P@{k}'] = round(topk.label.mean(),3)
            row[f'R@{k}'] = round(topk.label.sum()/npos,3)
        out.append(row)
    return pd.DataFrame(out), m
rep, _ = ranking_report(); rep

In [ ]:
# 8. Train final model on ALL data and export artifacts for the backend.
final = fit_xgb(X, y, X, y)   # full-data fit for deployment
ART = '/content/drive/MyDrive/AI Projects/model_artifacts'
os.makedirs(ART, exist_ok=True)
final.save_model(f'{ART}/xgb_disease_chemical.json')
json.dump(feature_cols, open(f'{ART}/feature_cols.json','w'))
json.dump(diseases, open(f'{ART}/diseases.json','w'))
feats.to_csv(f'{ART}/chemical_features.csv', index=False)
print('Saved model + feature_cols + diseases + chemical_features to', ART)

In [ ]:
# 9. disease -> chemical inference helper (used by the backend too)
def rank_chemicals_for_disease(disease, top=20):
    sub = feats.copy()
    for p in PROPS:
        sub[p] = pd.to_numeric(sub[p], errors='coerce').fillna(grid[p].median())
    oh = pd.DataFrame(0, index=sub.index, columns=[c for c in feature_cols if c.startswith('dis_')])
    col = f'dis_{disease}'
    if col in oh.columns: oh[col] = 1
    Xq = pd.concat([oh, sub[PROPS].reset_index(drop=True)], axis=1)[feature_cols]
    sub = sub.reset_index(drop=True)
    sub['score'] = final.predict_proba(Xq)[:,1]
    return sub.sort_values('score', ascending=False)[['ChemicalName','ChemicalID','score']].head(top)

rank_chemicals_for_disease('Asthma', 15)